[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/fedscope_new/blob/main/demo.ipynb)

# EHRI Federal Workforce Data Explorer

This notebook loads federal workforce data from HuggingFace and lets you explore trends over time:
- **Accessions** - New federal hires
- **Separations** - Federal employee departures
- **Employment** - Point-in-time workforce snapshots

**No authentication required** - all datasets are public. Available months are discovered automatically.

In [1]:
!pip install -q duckdb pandas plotly huggingface_hub great_tables

You should consider upgrading via the '/Users/abigailhaddad/Documents/repos/opm/venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import re
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from huggingface_hub import list_repo_files
from great_tables import GT
import base64
from IPython.display import HTML, display

def download_csv(df, filename):
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    display(HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}" style="background:#4CAF50;color:white;padding:8px 16px;text-decoration:none;border-radius:4px;display:inline-block;margin:10px 0;">Download {filename}</a>'))

## 1. Discover Available Files

We query the HuggingFace repo to find all available months automatically.

In [3]:
HF_REPO = "abigailhaddad/opm-federal-workforce"
BASE_URL = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main"

all_files = list(list_repo_files(HF_REPO, repo_type="dataset"))
parquet_files = [f for f in all_files if f.endswith(".parquet")]

def get_urls(data_type):
    """Get sorted list of HF URLs for a given data type.

    Handles versioned filenames (accessions_202511_v3.parquet).
    For each month, uses only the highest version available.
    """
    pattern = re.compile(rf"^{data_type}/{data_type}_(\d{{6}})_v(\d+)\.parquet$")
    # Also accept legacy unversioned files
    legacy = re.compile(rf"^{data_type}/{data_type}_(\d{{6}})\.parquet$")

    # {yyyymm: (version, path)}
    best = {}
    for f in parquet_files:
        m = pattern.match(f)
        if m:
            yyyymm, ver = m.group(1), int(m.group(2))
            if yyyymm not in best or ver > best[yyyymm][0]:
                best[yyyymm] = (ver, f)
            continue
        m = legacy.match(f)
        if m:
            yyyymm = m.group(1)
            if yyyymm not in best:
                best[yyyymm] = (0, f)

    months = sorted(best.keys())
    urls = [f"{BASE_URL}/{best[m][1]}" for m in months]
    return urls, months

acc_urls, acc_months = get_urls("accessions")
sep_urls, sep_months = get_urls("separations")
emp_urls, emp_months = get_urls("employment")

print(f"Accessions:  {len(acc_urls)} files  ({acc_months[0]} - {acc_months[-1]})")
print(f"Separations: {len(sep_urls)} files  ({sep_months[0]} - {sep_months[-1]})")
print(f"Employment:  {len(emp_urls)} files  ({emp_months[0]} - {emp_months[-1]})")

Accessions:  103 files  (201707 - 202601)
Separations: 93 files  (201805 - 202601)
Employment:  47 files  (202203 - 202601)


## 2. Load Accessions & Separations

These are small (~0.2 MB/month parquet) so we load all available months into DuckDB. Queries after this are instant.

In [4]:
%%time
N_MONTHS = 2  # most recent months per data type (6 files total)
db = duckdb.connect()

acc_list = ", ".join(f"'{u}'" for u in acc_urls[-N_MONTHS:])
db.execute(f"CREATE TABLE accessions AS SELECT * FROM read_parquet([{acc_list}])")

sep_list = ", ".join(f"'{u}'" for u in sep_urls[-N_MONTHS:])
db.execute(f"CREATE TABLE separations AS SELECT * FROM read_parquet([{sep_list}])")

emp_list = ", ".join(f"'{u}'" for u in emp_urls[-N_MONTHS:])
db.execute(f"CREATE VIEW employment AS SELECT * FROM read_parquet([{emp_list}])")

acc_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM accessions").fetchone()[0]
sep_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM separations").fetchone()[0]
emp_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM employment").fetchone()[0]
print(f"Loaded {N_MONTHS} months: {acc_months[-N_MONTHS]} - {acc_months[-1]}")
print(f"  Accessions: {acc_count:,} records")
print(f"  Separations: {sep_count:,} records")
print(f"  Employment: {emp_count:,} records")

Loaded 2 months: 202512 - 202601
  Accessions: 25,799 records
  Separations: 70,453 records
  Employment: 4,109,993 records
CPU times: user 1.6 s, sys: 125 ms, total: 1.73 s
Wall time: 3.25 s


## 3. Monthly Overview

In [5]:

# Summary table: accessions & separations totals per month
acc_totals = db.execute("""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as accessions
    FROM accessions GROUP BY month ORDER BY month
""").df()

sep_totals = db.execute("""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as separations
    FROM separations GROUP BY month ORDER BY month
""").df()

summary = acc_totals.merge(sep_totals, on='month')
summary['net'] = summary['accessions'] - summary['separations']
summary['Month'] = pd.to_datetime(summary['month'], format='%Y%m').dt.strftime('%B %Y')

display(
    GT(summary[['Month', 'accessions', 'separations', 'net']]
       .rename(columns={'accessions': 'Accessions', 'separations': 'Separations', 'net': 'Net Change'}))
    .tab_header(title="Federal Workforce: Monthly Overview")
    .fmt_integer(columns=['Accessions', 'Separations', 'Net Change'])
)
download_csv(summary[['Month', 'accessions', 'separations', 'net']], 'monthly_overview.csv')

GT(_tbl_data=             Month  Accessions  Separations  Net Change
0    December 2023        31.0         70.0       -39.0
1     January 2024        68.0         40.0        28.0
2    February 2024        62.0         31.0        31.0
3       March 2024        63.0         44.0        19.0
4       April 2024        40.0         13.0        27.0
5         May 2024        55.0         16.0        39.0
6        June 2024        60.0          9.0        51.0
7        July 2024        44.0         10.0        34.0
8      August 2024        49.0         13.0        36.0
9   September 2024        59.0          6.0        53.0
10    October 2024        58.0         17.0        41.0
11   November 2024        57.0         22.0        35.0
12   December 2024       104.0         15.0        89.0
13    January 2025       142.0         32.0       110.0
14   February 2025        55.0         35.0        20.0
15      March 2025        49.0        115.0       -66.0
16      April 2025       144.0         89.0        55.0
17        May 2025       287.0        173.0       114.0
18       June 2025       282.0        205.0        77.0
19       July 2025       228.0        324.0       -96.0
20     August 2025       313.0        461.0      -148.0
21  September 2025       529.0       6625.0     -6096.0
22    October 2025      1441.0       2933.0     -1492.0
23   November 2025      1078.0       2838.0     -1760.0
24   December 2025     10792.0      41365.0    -30573.0
25    January 2026      9709.0      14952.0     -5243.0, _body=<great_tables._gt_data.Body object at 0x1102a1f70>, _boxhead=Boxhead([ColInfo(var='Month', type=<ColInfoTypeEnum.default: 1>, column_label='Month', column_align='left', column_width=None), ColInfo(var='Accessions', type=<ColInfoTypeEnum.default: 1>, column_label='Accessions', column_align='right', column_width=None), ColInfo(var='Separations', type=<ColInfoTypeEnum.default: 1>, column_label='Separations', column_align='right', column_width=None), ColInfo(var='Net Change', type=<ColInfoTypeEnum.default: 1>, column_label='Net Change', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1102abca0>, _spanners=Spanners([]), _heading=Heading(title='Federal Workforce: Monthly Overview', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1102dd430>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1102ddb20>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1102ddd60>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1102ddbe0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(sc

In [6]:

# Accessions: 2-month comparison by agency
# Use the 2 most recent file months (not all dates in the data)
prev_m, curr_m = acc_months[-2], acc_months[-1]
prev_label = pd.to_datetime(prev_m, format='%Y%m').strftime('%B %Y')
curr_label = pd.to_datetime(curr_m, format='%Y%m').strftime('%B %Y')

acc_agency = db.execute(f"""
    SELECT agency,
           personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as count
    FROM accessions
    WHERE personnel_action_effective_date_yyyymm IN ('{prev_m}', '{curr_m}')
    GROUP BY agency, month
""").df()

pivot = (acc_agency
         .pivot(index='agency', columns='month', values='count')
         .fillna(0)
         .reset_index())
pivot.columns = ['Agency', prev_label, curr_label]
pivot['Change'] = (pivot[curr_label] - pivot[prev_label]).astype(int)
pivot['% Change'] = (pivot['Change'] / pivot[prev_label].replace(0, float('nan')) * 100).round(1)

top20 = pivot.reindex(pivot['Change'].abs().nlargest(20).index).reset_index(drop=True)

display(
    GT(top20)
    .tab_header(title=f"Accessions by Agency: {prev_label} vs {curr_label}",
                subtitle="Top 20 by absolute change")
    .fmt_integer(columns=[prev_label, curr_label, 'Change'])
    .fmt_number(columns=['% Change'], decimals=1)
)
download_csv(pivot.sort_values('Change', key=abs, ascending=False), 'accessions_comparison.csv')

GT(_tbl_data=                                      Agency  December 2025  January 2026  \
0            DEPARTMENT OF HOMELAND SECURITY         3855.0        1862.0   
1             DEPARTMENT OF VETERANS AFFAIRS         1939.0        2422.0   
2                      DEPARTMENT OF JUSTICE          206.0         595.0   
3                     DEPARTMENT OF COMMERCE          491.0         258.0   
4                  DEPARTMENT OF AGRICULTURE          152.0         373.0   
5                DEPARTMENT OF THE AIR FORCE          621.0         432.0   
6                     DEPARTMENT OF THE NAVY          633.0         752.0   
7   NAT AERONAUTICS AND SPACE ADMINISTRATION           10.0         115.0   
8                     DEPARTMENT OF TREASURY          429.0         332.0   
9    DEPARTMENT OF HEALTH AND HUMAN SERVICES           72.0         137.0   
10                    DEPARTMENT OF THE ARMY          473.0         428.0   
11                      DEPARTMENT OF ENERGY            9.0          49.0   
12              DEPARTMENT OF TRANSPORTATION          309.0         273.0   
13                       DEPARTMENT OF LABOR            4.0          35.0   
14                       DEPARTMENT OF STATE           39.0          18.0   
15        SECURITIES AND EXCHANGE COMMISSION            2.0          22.0   
16             SMALL BUSINESS ADMINISTRATION           19.0          35.0   
17                     DEPARTMENT OF DEFENSE         1160.0        1173.0   
18              ARMED FORCES RETIREMENT HOME            8.0           1.0   
19  DEPARTMENT OF HOUSING AND URBAN DEVELOPM            2.0           9.0   

    Change  % Change  
0    -1993     -51.7  
1      483      24.9  
2      389     188.8  
3     -233     -47.5  
4      221     145.4  
5     -189     -30.4  
6      119      18.8  
7      105    1050.0  
8      -97     -22.6  
9       65      90.3  
10     -45      -9.5  
11      40     444.4  
12     -36     -11.7  
13      31     775.0  
14     -21     -53.8  
15      20    1000.0  
16      16      84.2  
17      13       1.1  
18      -7     -87.5  
19       7     350.0  , _body=<great_tables._gt_data.Body object at 0x11028d940>, _boxhead=Boxhead([ColInfo(var='Agency', type=<ColInfoTypeEnum.default: 1>, column_label='Agency', column_align='left', column_width=None), ColInfo(var='December 2025', type=<ColInfoTypeEnum.default: 1>, column_label='December 2025', column_align='right', column_width=None), ColInfo(var='January 2026', type=<ColInfoTypeEnum.default: 1>, column_label='January 2026', column_align='right', column_width=None), ColInfo(var='Change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='right', column_width=None), ColInfo(var='% Change', type=<ColInfoTypeEnum.default: 1>, column_label='% Change', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x110385430>, _spanners=Spanners([]), _heading=Heading(title='Accessions by Agency: December 2025 vs January 2026', subtitle='Top 20 by absolute change', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1103c5940>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1103c59a0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1103c5a00>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1103c5b50>, <great_tables._gt_data.FormatInfo object at 0x1103c5d00>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table

## 4. Accessions by Agency — Heatmap

In [7]:

# Heatmap: top 15 agencies by accessions across loaded months
months_filter = ", ".join(f"'{m}'" for m in acc_months[-N_MONTHS:])
acc_heat = db.execute(f"""
    SELECT agency,
           personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as count
    FROM accessions
    WHERE personnel_action_effective_date_yyyymm IN ({months_filter})
    GROUP BY agency, month
""").df()

month_label_map = {m: pd.to_datetime(m, format='%Y%m').strftime('%b %Y') for m in acc_months[-N_MONTHS:]}
top15 = acc_heat.groupby('agency')['count'].sum().nlargest(15).index.tolist()

heat = (acc_heat[acc_heat['agency'].isin(top15)]
        .assign(month_label=lambda d: d['month'].map(month_label_map))
        .pivot(index='agency', columns='month_label', values='count')
        .fillna(0))
heat = heat[list(month_label_map.values())]  # ensure column order
heat = heat.sort_values(heat.columns[-1], ascending=False)

fig = px.imshow(heat, text_auto=',d', color_continuous_scale='Blues',
                title='Accessions — Top 15 Agencies',
                labels=dict(x='Month', y='Agency', color='Hires'))
fig.update_layout(height=520, xaxis_title='', yaxis_title='', coloraxis_showscale=False)
fig.show()

## 5. Columns & Available Values

In [8]:
print("Accessions columns:", db.execute("DESCRIBE accessions").df()['column_name'].tolist())
print("\nSeparations columns:", db.execute("DESCRIBE separations").df()['column_name'].tolist())

Accessions columns: ['accession_category', 'accession_category_code', 'age_bracket', 'agency', 'agency_code', 'agency_subelement', 'agency_subelement_code', 'annualized_adjusted_basic_pay', 'appointment_not_to_exceed_date', 'appointment_type', 'appointment_type_code', 'bargaining_unit', 'bargaining_unit_code', 'bargaining_unit_status', 'cfo_act_agency_indicator', 'consolidated_statistical_area', 'consolidated_statistical_area_code', 'core_based_statistical_area', 'core_based_statistical_area_code', 'count', 'duty_station_code', 'duty_station_country', 'duty_station_country_code', 'duty_station_county', 'duty_station_county_code', 'duty_station_state', 'duty_station_state_abbreviation', 'duty_station_state_code', 'duty_station_state_country_territory_code', 'education_level', 'education_level_bracket', 'education_level_code', 'flsa_category', 'flsa_category_code', 'grade', 'length_of_service_years', 'locality_pay_area', 'locality_pay_area_code', 'nsftp_indicator', 'occupational_category

In [9]:
for field in ['agency', 'occupational_group', 'education_level', 'duty_station_state']:
    result = db.execute(f"""
        SELECT {field} as value, SUM(CAST(count AS INTEGER)) as n
        FROM accessions WHERE {field} IS NOT NULL AND {field} != ''
        GROUP BY {field} ORDER BY n DESC LIMIT 8
    """).df()
    print(f"\n{field}:")
    for _, row in result.iterrows():
        print(f"  {row['value']}: {row['n']:,}")


agency:
  DEPARTMENT OF VETERANS AFFAIRS: 6,647.0
  DEPARTMENT OF HOMELAND SECURITY: 5,937.0
  DEPARTMENT OF DEFENSE: 2,917.0
  DEPARTMENT OF THE NAVY: 1,573.0
  DEPARTMENT OF THE ARMY: 1,391.0
  DEPARTMENT OF THE AIR FORCE: 1,368.0
  DEPARTMENT OF JUSTICE: 954.0
  DEPARTMENT OF INTERIOR: 942.0

occupational_group:
  INVESTIGATION GROUP: 5,675.0
  MEDICAL, HOSPITAL, DENTAL, AND PUBLIC HEALTH GROUP: 4,776.0
  GENERAL ADMINISTRATIVE, CLERICAL, AND OFFICE SERVICES GROUP: 2,174.0
  MISCELLANEOUS OCCUPATIONS GROUP: 1,301.0
  LEGAL AND KINDRED GROUP: 1,299.0
  SOCIAL SCIENCE, PSYCHOLOGY, AND WELFARE GROUP: 942.0
  GENERAL SERVICES AND SUPPORT WORK FAMILY: 839.0
  EDUCATION GROUP: 804.0

education_level:
  HIGH SCHOOL GRADUATE OR CERTIFICATE OF EQUIVALENCY: 11,076.0
  BACHELOR'S DEGREE: 5,473.0
  MASTER'S DEGREE: 2,948.0
  ASSOCIATE DEGREE: 1,499.0
  FIRST PROFESSIONAL: 945.0
  DOCTORATE DEGREE: 739.0
  SOME COLLEGE - LESS THAN ONE YEAR: 650.0
  ONE YEAR COLLEGE: 427.0

duty_station_state:
 

In [10]:
%%time
N_MONTHS = 2
recent_emp_urls = emp_urls[-N_MONTHS:]
recent_emp_months = emp_months[-N_MONTHS:]
print(f"Loading employment: {recent_emp_months[0]} - {recent_emp_months[-1]}")

emp_list = ", ".join(f"'{u}'" for u in recent_emp_urls)
db.execute(f"CREATE OR REPLACE VIEW employment AS SELECT * FROM read_parquet([{emp_list}])")

emp_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM employment").fetchone()[0]
print(f"Loaded {emp_count:,} employee records")
print("Columns:", db.execute("DESCRIBE employment").df()['column_name'].tolist())

Loading employment: 202512 - 202601


Loaded 4,109,993 employee records
Columns: ['age_bracket', 'agency', 'agency_code', 'agency_subelement', 'agency_subelement_code', 'annualized_adjusted_basic_pay', 'appointment_type', 'appointment_type_code', 'bargaining_unit', 'bargaining_unit_code', 'bargaining_unit_status', 'cfo_act_agency_indicator', 'consolidated_statistical_area', 'consolidated_statistical_area_code', 'core_based_statistical_area', 'core_based_statistical_area_code', 'count', 'duty_station_code', 'duty_station_country', 'duty_station_country_code', 'duty_station_county', 'duty_station_county_code', 'duty_station_state', 'duty_station_state_abbreviation', 'duty_station_state_code', 'duty_station_state_country_territory_code', 'education_level', 'education_level_bracket', 'education_level_code', 'flsa_category', 'flsa_category_code', 'grade', 'length_of_service_years', 'locality_pay_area', 'locality_pay_area_code', 'nsftp_indicator', 'occupational_category', 'occupational_category_code', 'occupational_group', 'occu

In [11]:

# Employment snapshot: top 20 agencies in the most recent month
latest_month = emp_months[-1]
latest_label = pd.to_datetime(latest_month, format='%Y%m').strftime('%B %Y')

top_emp = db.execute(f"""
    SELECT agency, SUM(CAST(count AS INTEGER)) as employees
    FROM employment WHERE snapshot_yyyymm = '{latest_month}'
    GROUP BY agency ORDER BY employees DESC LIMIT 20
""").df()

fig = px.bar(
    top_emp, x='employees', y='agency', orientation='h',
    title=f'Top 20 Agencies by Employment — {latest_label}',
    color='employees', color_continuous_scale='Blues',
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'},
                  xaxis=dict(tickformat=','), showlegend=False, coloraxis_showscale=False)
fig.show()
download_csv(top_emp, f'employment_{latest_month}.csv')

## 7. Try Your Own Queries

Three DuckDB tables are loaded: `accessions`, `separations`, `employment`.

**Common fields:**
- `agency` — e.g. `DEPARTMENT OF DEFENSE`
- `duty_station_state` — e.g. `CALIFORNIA`
- `occupational_group` — e.g. `INFORMATION TECHNOLOGY GROUP`
- `occupational_series` — e.g. `2210`
- `education_level`, `age_bracket`, `supervisory_status`
- `personnel_action_effective_date_yyyymm` (accessions/separations)
- `snapshot_yyyymm` (employment)
- `count` — number of people in that combination of values

All counts are stored as strings — cast with `CAST(count AS INTEGER)`.

In [12]:

# Example: Army separations in the most recent month
latest_sep_month = sep_months[-1]
latest_sep_label = pd.to_datetime(latest_sep_month, format='%Y%m').strftime('%B %Y')

df = db.execute(f"""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as departures
    FROM separations
    WHERE agency = 'DEPARTMENT OF THE ARMY'
      AND personnel_action_effective_date_yyyymm = '{latest_sep_month}'
    GROUP BY month ORDER BY month
""").df()
print(f"Army separations in {latest_sep_label}: {df['departures'].sum():,}")

Army separations in January 2026: 1,039.0
